# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof. Luis Eduardo Falcón Morales

### **Actividad en Equipos: Chatbot RAG + LLM**

---

# 💜 **Seguras 2.0** — Asistente contra el ciberacoso (RAG + LLM)

* **Nombres y matrículas:**
  *   _Nombre 1 — Matrícula_
  *   _Nombre 2 — Matrícula_
  *   _Nombre 3 — Matrícula_

* **Número de Equipo:** _____

---
> **Cómo ejecutar este cuaderno (Google Colab):**
> 1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
> 2. Ejecuta las celdas en orden (la primera instala dependencias).
> 3. La última celda abre la **interfaz Gradio** con un enlace público (`share=True`).


## 1. Introducción y justificación de la problemática

El **ciberacoso** y la **violencia digital** afectan de forma desproporcionada a mujeres y
adolescentes: mensajes hostiles, amenazas, hostigamiento, suplantación de identidad,
*doxing* y difusión de contenido íntimo sin consentimiento. Muchas víctimas **no saben
qué hacer, a quién acudir ni cómo reunir evidencia**, y con frecuencia sienten miedo o
vergüenza de pedir ayuda.

Este proyecto extiende un trabajo previo del equipo, **«Seguras1»**, que era un
**clasificador clásico** (TF-IDF + Regresión Logística) entrenado sobre el dataset
[`Dataset-Acoso-Twitter-Es`](https://huggingface.co/datasets/somosnlp-hackathon-2022/Dataset-Acoso-Twitter-Es)
para decir si un texto *es o no* ciberacoso. Ese modelo respondía **sí/no**, pero no
**orientaba** a la persona.

**Seguras 2.0** convierte ese detector en una pieza de un **chatbot conversacional RAG + LLM**:

| | Seguras1 (antes) | Seguras 2.0 (ahora) |
|---|---|---|
| Tarea | Clasificar acoso sí/no | Acompañar y orientar conversando |
| Salida | Etiqueta | Respuesta empática + pasos + recursos |
| Conocimiento | Solo el dataset | Base documental en español (RAG) |
| Modelo | Regresión Logística | LLM cuantizado (NF4 / estilo QLoRA) |
| El clasificador… | era el producto final | es una **herramienta interna** del chatbot |

**¿Por qué RAG + LLM?** Un LLM por sí solo puede *alucinar* leyes, teléfonos o
procedimientos. Con **Recuperación Aumentada Generativa (RAG)** el modelo responde
**fundamentado en documentos curados en español** (definiciones, pasos de actuación,
recursos verificados de México, marco legal), lo que **reduce las alucinaciones** y
**personaliza** las respuestas a la temática de seguridad frente al ciberacoso.


## 2. Arquitectura del sistema

```
                         ┌──────────────────────────────────────────────┐
   Mensaje de la         │                 SEGURAS 2.0                    │
   persona  ───────────► │                                                │
                         │  (A) Detector  TF-IDF + LogReg  → señal acoso  │
                         │  (B) Embeddings multilingües (consulta)        │
                         │  (C) FAISS  → recupera top-k documentos (RAG)  │
                         │  (D) Prompt = persona + contexto + señal + hist│
                         │  (E) LLM cuantizado NF4 (estilo QLoRA)         │
                         │        └─► genera respuesta en español         │
                         └──────────────────────────────────────────────┘
                                          │
   Interfaz Gradio  ◄─────────────────────┘  (respuesta + detección + fuentes)
```

**Componentes:**
- **(A) Detector** — el modelo de *Seguras1* reutilizado como *tool* que aporta una señal
  («posible acoso» / «sin señales»).
- **(B) Embeddings** — `paraphrase-multilingual-MiniLM-L12-v2` (entiende español).
- **(C) Recuperador** — índice **FAISS** por similitud coseno sobre la base de conocimiento.
- **(D) Orquestador** — arma el *prompt* con la persona del bot, el contexto recuperado,
  la señal del detector y el historial.
- **(E) LLM** — modelo *instruct* cargado en **4 bits NF4** con **doble cuantización** y
  **cómputo en bfloat16** (la técnica de cuantización vista en la teoría de QLoRA).


## 3. Instalación de dependencias

In [ ]:
# Ejecuta una sola vez. En Colab activa GPU (T4) antes de correr.
!pip -q install "gradio>=5.49,<6" "transformers>=4.44" "accelerate>=0.33" \
                "bitsandbytes>=0.43" "sentence-transformers>=3.0" "faiss-cpu>=1.8" \
                "datasets>=2.20" "scikit-learn" "nltk" "joblib"
print("✅ Dependencias instaladas. Si Colab lo pide, reinicia el entorno y re-ejecuta desde aquí.")

## 4. Importaciones y configuración

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

# ===== Configuración =====
# LLM potente, abierto y excelente en español (sin gating en Hugging Face).
#   • Más potencia:   "Qwen/Qwen2.5-7B-Instruct"   (cabe en T4 en 4-bit, más lento)
#   • Alternativa:    "meta-llama/Llama-3.2-3B-Instruct"  (requiere token + aceptar licencia)
MODEL_ID       = "Qwen/Qwen2.5-3B-Instruct"
EMBED_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TOP_K          = 4   # documentos recuperados por consulta

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE,
      "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "— (usa GPU para mejor desempeño)")

## 5. Datos: dataset de acoso en Twitter (español)

Usamos el mismo dataset de *Seguras1*: [`Dataset-Acoso-Twitter-Es`](https://huggingface.co/datasets/somosnlp-hackathon-2022/Dataset-Acoso-Twitter-Es).
La columna `text` contiene el tuit y `task1` la etiqueta de acoso.

In [ ]:
DATA_URL = "hf://datasets/somosnlp-hackathon-2022/Dataset-Acoso-Twitter-Es/datasetfinal.csv"
try:
    df = pd.read_csv(DATA_URL)
except Exception as e:
    print("read_csv hf:// no disponible, usando datasets.load_dataset()...", repr(e))
    from datasets import load_dataset
    ds = load_dataset("somosnlp-hackathon-2022/Dataset-Acoso-Twitter-Es")
    split = "train" if "train" in ds else list(ds.keys())[0]
    df = ds[split].to_pandas()

print("Filas, columnas:", df.shape)
df.head()

## 6. Preprocesamiento de texto

In [ ]:
import nltk
from nltk.corpus import stopwords
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

STOP_ES = set(stopwords.words("spanish"))

def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)        # URLs
    text = re.sub(r"@\w+", " ", text)                     # menciones
    text = re.sub(r"#\w+", " ", text)                     # hashtags
    text = re.sub(r"\d+", " ", text)                      # números
    text = re.sub(r"[^a-záéíóúñü\s]", " ", text.lower())  # puntuación / símbolos
    text = " ".join(w for w in text.split() if w not in STOP_ES)
    return text.strip()

print(clean_text("¡@usuario eres una INÚTIL!! nadie te quiere 😤 http://x.co #odio"))

## 7. Entrenamiento del detector (herramienta interna)

Reutilizamos el enfoque de *Seguras1* (TF-IDF + Regresión Logística), con pequeñas mejoras
(*n-gramas* 1–2 y `class_weight='balanced'`). Este modelo **no** es el chatbot: será una
**señal** que el chatbot considera al responder.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import joblib

data = df[["text", "task1"]].dropna().copy()
data["text_clean"] = data["text"].apply(clean_text)

le = LabelEncoder()
data["y"] = le.fit_transform(data["task1"].astype(str))
print("Clases en task1:", list(le.classes_))

# Detectar dinámicamente qué etiqueta(s) representan ACOSO
ACOSO_CLASSES = [c for c in le.classes_
                 if "acoso" in c.lower() and not c.lower().replace("_", " ").strip().startswith("no")]
if not ACOSO_CLASSES:
    ACOSO_CLASSES = [c for c in le.classes_ if c.lower() in {"1", "si", "sí", "yes", "true"}]
print("Etiqueta(s) de acoso:", ACOSO_CLASSES)

Xtr, Xte, ytr, yte = train_test_split(
    data["text_clean"], data["y"], test_size=0.2, random_state=SEED, stratify=data["y"])

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
Xtr_v, Xte_v = vectorizer.fit_transform(Xtr), vectorizer.transform(Xte)

clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
clf.fit(Xtr_v, ytr)

y_pred = clf.predict(Xte_v)
print("\nAccuracy:", round(accuracy_score(yte, y_pred), 4))
print(classification_report(yte, y_pred, target_names=[str(c) for c in le.classes_]))

joblib.dump(clf, "seguras_clf.pkl")
joblib.dump(vectorizer, "seguras_tfidf.pkl")
joblib.dump(le, "seguras_le.pkl")
print("✅ Detector entrenado y guardado.")

In [ ]:
ACOSO_IDX = set(le.transform(ACOSO_CLASSES)) if ACOSO_CLASSES else set()

def detectar_acoso(texto: str) -> dict:
    '''Señal del detector: ¿el texto parece acoso? + probabilidad.'''
    v = vectorizer.transform([clean_text(texto)])
    pred = int(clf.predict(v)[0])
    proba = clf.predict_proba(v)[0]
    p_acoso = float(sum(proba[i] for i in ACOSO_IDX)) if ACOSO_IDX else float(proba.max())
    return {
        "es_acoso": pred in ACOSO_IDX,
        "etiqueta": str(le.inverse_transform([pred])[0]),
        "prob_acoso": round(p_acoso, 3),
    }

print(detectar_acoso("eres una inútil, nadie te quiere, deberías desaparecer"))
print(detectar_acoso("¡felicidades por tu nuevo trabajo, me da mucho gusto!"))

## 8. Sistema RAG — Base de conocimiento

Documentos **curados en español** sobre ciberacoso: qué es, tipos, cómo actuar, recursos
**verificados de México** y marco legal. Esta base es la que «fundamenta» las respuestas
del LLM.

> ⚠️ Los teléfonos incluidos fueron verificados (México). Aun así, ante una emergencia real
> siempre conviene confirmar el número vigente con fuentes oficiales.

In [ ]:
KB = [
    {"id": "def-01", "tema": "Qué es el ciberacoso",
     "texto": "El ciberacoso (o acoso digital) es una conducta repetida e intencional que busca "
              "humillar, intimidar, amenazar o dañar a una persona a través de medios digitales: "
              "redes sociales, mensajería, correo o videojuegos. A diferencia de una discusión puntual, "
              "se caracteriza por la repetición y el desequilibrio de poder."},
    {"id": "tipos-01", "tema": "Tipos de ciberacoso",
     "texto": "Formas frecuentes: hostigamiento y mensajes ofensivos repetidos; amenazas; suplantación "
              "de identidad; doxing (publicar datos personales sin permiso); ciberacecho (stalking) o "
              "vigilancia constante; y difusión de imágenes o videos íntimos sin consentimiento (sextorsión)."},
    {"id": "senales-01", "tema": "Señales de alerta",
     "texto": "Recibir insultos o amenazas constantes, que compartan información o fotos tuyas sin permiso, "
              "que te etiqueten o mencionen para ridiculizarte, mensajes de cuentas falsas, presión para "
              "enviar contenido íntimo, o que alguien controle con quién hablas o dónde estás."},
    {"id": "actuar-01", "tema": "Qué hacer: no respondas y resguarda evidencia",
     "texto": "1) Evita responder o seguir la provocación. 2) NO borres los mensajes: son evidencia. "
              "3) Guarda capturas de pantalla incluyendo usuario, fecha, hora y la URL o enlace. "
              "Esta evidencia es clave para reportar y, si decides, denunciar."},
    {"id": "actuar-02", "tema": "Qué hacer: bloquear, reportar y contar a alguien",
     "texto": "Tras resguardar la evidencia: bloquea o restringe a la persona; reporta la cuenta o el "
              "contenido en la propia plataforma (todas tienen opción de reportar acoso); ajusta tu "
              "privacidad; y cuéntale lo que pasa a alguien de confianza para no enfrentarlo en soledad."},
    {"id": "actuar-03", "tema": "Cómo guardar evidencia correctamente",
     "texto": "Captura la conversación completa (no fragmentos), asegúrate de que se vea el nombre de "
              "usuario y la fecha, guarda los enlaces (URLs) de las publicaciones y respáldalo en un lugar "
              "seguro (correo o nube). Si es posible, pide a un testigo que también vea el contenido."},
    {"id": "recursos-mx-01", "tema": "Recursos de ayuda en México (líneas)",
     "texto": "En México puedes llamar a la Línea para Mujeres en Situación de Violencia 800 108 4053, "
              "disponible 24 horas, todos los días, con apoyo psicológico y asesoría jurídica de primer "
              "contacto. Para denunciar violencia de género puedes marcar 079 y elegir la opción 1. "
              "Ante una emergencia o riesgo inmediato, llama al 911."},
    {"id": "recursos-mx-02", "tema": "Dónde denunciar formalmente",
     "texto": "Para iniciar una denuncia formal puedes acudir al Ministerio Público o a la Fiscalía de tu "
              "entidad; varias cuentan con unidades especializadas en delitos digitales o de género. Lleva "
              "tu evidencia (capturas, enlaces, fechas). La asesoría de las líneas de atención te orienta "
              "sobre el procedimiento según tu estado."},
    {"id": "legal-01", "tema": "Marco legal: Ley Olimpia (violencia digital)",
     "texto": "En México, la llamada Ley Olimpia es un conjunto de reformas que reconocen la violencia "
              "digital y sancionan la difusión de contenido íntimo sexual sin consentimiento. Permite que "
              "estos actos sean considerados delito y da sustento para denunciar. Los detalles y penas "
              "pueden variar según el código penal de cada estado."},
    {"id": "emocional-01", "tema": "Apoyo emocional y autocuidado",
     "texto": "Lo que sientes es válido y no es tu culpa: la responsabilidad es de quien acosa. Date "
              "permiso de pedir apoyo, habla con personas de confianza y considera acompañamiento "
              "psicológico. Tomar pausas de las redes y cuidar tu descanso también ayuda a recuperarte."},
    {"id": "seguridad-01", "tema": "Seguridad digital preventiva",
     "texto": "Usa contraseñas largas y distintas por cuenta, activa la verificación en dos pasos (2FA), "
              "revisa la privacidad de tus perfiles, limita quién ve tu ubicación y desconfía de enlaces o "
              "solicitudes de personas desconocidas."},
    {"id": "mito-01", "tema": "Mitos sobre el ciberacoso",
     "texto": "Mito: 'si lo ignoras, desaparece'. No siempre: ignorar no sustituye reportar y resguardar "
              "evidencia. Mito: 'es solo en internet, no es grave'. Falso: el daño emocional es real y "
              "puede escalar. Mito: 'te lo buscaste'. Nunca: la culpa es de quien acosa, no de la víctima."},
    {"id": "menores-01", "tema": "Cuando involucra a menores de edad",
     "texto": "Si el acoso involucra a una persona menor de edad, es importante avisar a madres, padres o "
              "tutores y a la escuela, además de reportar y, cuando corresponda, denunciar ante las "
              "autoridades. La protección de menores es prioritaria."},
    {"id": "alcance-01", "tema": "Qué es y qué no es Seguras",
     "texto": "Seguras es un asistente orientativo que da información y primeros pasos. NO sustituye la "
              "atención profesional psicológica, jurídica ni a las autoridades, y no puede garantizar "
              "resultados. Ante riesgo a la integridad, lo prioritario es contactar a las líneas de ayuda y al 911."},
]
print("Documentos base:", len(KB))

In [ ]:
# Enriquecemos la base con ejemplos REALES del dataset (uso adicional de los datos)
def _ejemplos(es_acoso, n=3):
    mask = data["y"].isin(list(ACOSO_IDX)) if es_acoso else ~data["y"].isin(list(ACOSO_IDX))
    textos = data.loc[mask, "text"].dropna().astype(str)
    textos = textos[textos.str.len().between(15, 180)].head(n).tolist()
    return textos

for i, t in enumerate(_ejemplos(True)):
    KB.append({"id": f"ej-acoso-{i}", "tema": "Ejemplo etiquetado como ACOSO (dataset)",
               "texto": f"Ejemplo real del dataset clasificado como acoso: «{t}»"})
for i, t in enumerate(_ejemplos(False)):
    KB.append({"id": f"ej-no-{i}", "tema": "Ejemplo etiquetado como NO acoso (dataset)",
               "texto": f"Ejemplo real del dataset clasificado como NO acoso: «{t}»"})

print("Total documentos en la base de conocimiento:", len(KB))

## 9. Embeddings + índice vectorial (FAISS)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer(EMBED_MODEL_ID, device=DEVICE)
KB_TEXTS = [f"{d['tema']}. {d['texto']}" for d in KB]

emb = embedder.encode(KB_TEXTS, normalize_embeddings=True,
                      convert_to_numpy=True, show_progress_bar=False).astype("float32")

index = faiss.IndexFlatIP(emb.shape[1])   # producto interno = coseno (vectores normalizados)
index.add(emb)
print(f"✅ Índice FAISS: {index.ntotal} vectores de dimensión {emb.shape[1]}")

In [ ]:
def recuperar(consulta: str, k: int = TOP_K):
    '''Devuelve los k documentos más similares a la consulta.'''
    q = embedder.encode([consulta], normalize_embeddings=True,
                        convert_to_numpy=True).astype("float32")
    scores, idx = index.search(q, k)
    return [{"tema": KB[i]["tema"], "texto": KB[i]["texto"], "score": round(float(s), 3)}
            for s, i in zip(scores[0], idx[0])]

for f in recuperar("me están difundiendo fotos íntimas sin mi permiso, ¿qué hago?"):
    print(f["score"], "—", f["tema"])

## 10. LLM cuantizado en NF4 (estilo QLoRA)

Cargamos el LLM en **4 bits con formato NF4**, **doble cuantización** y **cómputo en
`bfloat16`**, tal como se explicó en la teoría de **QLoRA**. Esto permite ejecutar un
modelo *instruct* potente dentro de la VRAM de una GPU T4 de Colab.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if DEVICE == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                      # pesos en 4 bits
        bnb_4bit_quant_type="nf4",              # formato NF4 (normal float 4)
        bnb_4bit_use_double_quant=True,         # doble cuantización (ahorra VRAM extra)
        bnb_4bit_compute_dtype=torch.bfloat16,  # cómputo en BF16 para estabilidad
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto")
else:
    print("⚠️ Sin GPU: se cargará en CPU (lento). En Colab activa GPU T4 para usar NF4.")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32, device_map="cpu")

model.eval()
print("✅ LLM cargado:", MODEL_ID)

## 11. Orquestación: detector + RAG + LLM = chatbot

In [ ]:
SYSTEM_PROMPT = (
    "Eres «Seguras», una asistente virtual empática y respetuosa, especializada en orientar a "
    "mujeres y personas que enfrentan ciberacoso o violencia digital en México.\n"
    "Reglas que SIEMPRE sigues:\n"
    "1. Responde en español, con calidez y sin juzgar; nunca minimices lo que la persona siente "
    "ni la culpes.\n"
    "2. Fundamenta tu respuesta en el CONTEXTO recuperado. Si el contexto no alcanza, dilo con "
    "honestidad y da orientación general prudente; NO inventes leyes, teléfonos ni procedimientos.\n"
    "3. Ofrece pasos concretos y accionables (resguardar evidencia, reportar, bloquear, denunciar) "
    "cuando la situación lo amerite.\n"
    "4. Si percibes riesgo o angustia, recuerda con tacto los recursos de ayuda y la importancia de "
    "apoyarse en personas de confianza y autoridades. No prometas confidencialidad absoluta de las líneas.\n"
    "5. Eres un apoyo ORIENTATIVO: no sustituyes atención profesional, psicológica ni legal.\n"
    "6. Sé clara y breve (máximo ~6 oraciones o una lista corta de pasos)."
)

def construir_mensajes(message, history, deteccion, fuentes):
    contexto = "\n".join(f"- ({f['tema']}) {f['texto']}" for f in fuentes)
    pista = (
        "Señal del detector automático sobre el ÚLTIMO mensaje de la persona: "
        + ("POSIBLE ACOSO" if deteccion["es_acoso"] else "SIN señales claras de acoso")
        + f" (prob. acoso ≈ {deteccion['prob_acoso']}). Tómalo como una pista, no como verdad absoluta."
    )
    sys = f"{SYSTEM_PROMPT}\n\n### CONTEXTO RECUPERADO (RAG)\n{contexto}\n\n### {pista}"
    msgs = [{"role": "system", "content": sys}]
    for h in history[-6:]:                       # memoria conversacional corta
        msgs.append({"role": h["role"], "content": h["content"]})
    msgs.append({"role": "user", "content": message})
    return msgs

@torch.inference_mode()
def generar(message, history, deteccion, fuentes, max_new_tokens=350):
    msgs = construir_mensajes(message, history, deteccion, fuentes)
    enc = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=True,
        temperature=0.7, top_p=0.9, repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id)
    texto = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return texto.strip()

print("✅ Orquestador listo.")

## 12. Pruebas rápidas (sin interfaz)

In [ ]:
def responder_demo(pregunta, history=None):
    history = history or []
    det = detectar_acoso(pregunta)
    fts = recuperar(pregunta, TOP_K)
    resp = generar(pregunta, history, det, fts)
    print("👤 Usuaria:", pregunta)
    print(f"🔎 Detector: {'⚠️ posible acoso' if det['es_acoso'] else '✅ sin señales'} "
          f"(prob={det['prob_acoso']})")
    print("🤖 Seguras:", resp)
    print("📚 Fuentes RAG:", [f["tema"] for f in fts])
    print("-" * 95)
    return resp

preguntas = [
    "¿Qué es el ciberacoso?",
    "Un excompañero me amenaza por mensajes y dice que va a publicar fotos mías, ¿qué hago?",
    "Tengo mucho miedo, ¿a dónde puedo llamar en México ahora mismo?",
    "Siento que es mi culpa por haber confiado en él...",
]
for q in preguntas:
    responder_demo(q)

## 13. Interfaz coqueta en Gradio 💜

In [ ]:
import gradio as gr

# Avatar del bot (SVG embebido: no depende de internet)
BOT_AVATAR = "data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 100 100'%3E%3Cdefs%3E%3ClinearGradient id='g' x1='0' y1='0' x2='1' y2='1'%3E%3Cstop offset='0' stop-color='%23c084fc'/%3E%3Cstop offset='1' stop-color='%23f472b6'/%3E%3C/linearGradient%3E%3C/defs%3E%3Ccircle cx='50' cy='50' r='50' fill='url(%23g)'/%3E%3Ctext x='50' y='66' font-size='52' text-anchor='middle' fill='white'%3ES%3C/text%3E%3C/svg%3E"

tema = gr.themes.Soft(
    primary_hue="purple", secondary_hue="pink", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Quicksand"), "ui-sans-serif", "sans-serif"],
)

HEADER = """
<div style="background:linear-gradient(120deg,#a855f7 0%,#ec4899 100%);
            border-radius:22px;padding:26px 28px;color:white;
            box-shadow:0 10px 30px rgba(168,85,247,.35);text-align:center;">
  <div style="font-size:30px;font-weight:800;letter-spacing:.5px;">💜 Seguras</div>
  <div style="font-size:16px;opacity:.95;margin-top:6px;">
     Tu aliada contra el ciberacoso · orientación, pasos y recursos · 100&#37; en español
  </div>
</div>
"""

FOOTER = """
<div style="margin-top:14px;padding:14px 18px;border-radius:16px;
            background:#faf5ff;border:1px solid #efd9fb;font-size:13px;color:#6b21a8;">
  <b>📞 Si estás en riesgo, pide ayuda ahora:</b>
  Línea Mujeres <b>800 108 4053</b> (24 h) · Denuncia <b>079 opción 1</b> · Emergencias <b>911</b>.<br>
  <span style="color:#7c3aed;">Seguras es un apoyo orientativo y no sustituye atención profesional, legal ni a las autoridades.</span>
</div>
"""

def panel_inicial():
    return ("<div style='padding:14px;border-radius:16px;background:#fdf2f8;"
            "border:1px solid #fbcfe8;color:#9d174d;font-size:14px;'>"
            "<b>🔎 Análisis</b><br>Escribe un mensaje para ver aquí la detección y las "
            "fuentes consultadas. 💌</div>")

def render_panel(det, fuentes):
    color = "#e11d48" if det["es_acoso"] else "#16a34a"
    badge = "⚠️ Posible acoso" if det["es_acoso"] else "✅ Sin señales claras"
    chips = "".join(
        f"<div style='margin:6px 0;padding:8px 10px;border-radius:12px;background:#f5f3ff;"
        f"border:1px solid #e9d5ff;color:#5b21b6;font-size:12.5px;'>"
        f"<b>{f['tema']}</b><br><span style='color:#7c3aed;'>similitud {f['score']}</span></div>"
        for f in fuentes)
    return (
        "<div style='padding:14px;border-radius:16px;background:#fdf2f8;border:1px solid #fbcfe8;'>"
        f"<div style='font-weight:700;color:#9d174d;'>🔎 Análisis del último mensaje</div>"
        f"<div style='margin:10px 0;padding:10px;border-radius:12px;background:white;"
        f"border:1px solid #fbcfe8;color:{color};font-weight:700;'>{badge} "
        f"<span style='color:#6b7280;font-weight:500;'>(prob {det['prob_acoso']})</span></div>"
        f"<div style='font-weight:700;color:#9d174d;margin-top:6px;'>📚 Fuentes consultadas (RAG)</div>"
        f"{chips}</div>")

def chat_fn(message, history):
    if not message or not message.strip():
        return "", history, panel_inicial()
    det = detectar_acoso(message)
    fts = recuperar(message, TOP_K)
    resp = generar(message, history, det, fts)
    history = history + [{"role": "user", "content": message},
                         {"role": "assistant", "content": resp}]
    return "", history, render_panel(det, fts)

EJEMPLOS = [
    "¿Qué es el ciberacoso y cómo sé si me está pasando?",
    "Alguien creó una cuenta falsa con mis fotos, ¿qué hago?",
    "Me amenazan con difundir fotos íntimas mías.",
    "Tengo miedo, ¿a dónde puedo llamar en México?",
]

with gr.Blocks(theme=tema, title="Seguras 💜") as demo:
    gr.HTML(HEADER)
    with gr.Row():
        with gr.Column(scale=7):
            chatbot = gr.Chatbot(type="messages", height=440, show_label=False,
                                 avatar_images=(None, BOT_AVATAR),
                                 placeholder="<center>💜<br>Estoy aquí para acompañarte.<br>"
                                             "Cuéntame qué está pasando.</center>")
            with gr.Row():
                txt = gr.Textbox(placeholder="Cuéntame qué está pasando... 💬",
                                 show_label=False, scale=8, lines=1, autofocus=True)
                send = gr.Button("Enviar 💌", variant="primary", scale=1, min_width=110)
            gr.Examples(EJEMPLOS, inputs=txt, label="Ejemplos para empezar")
            clear = gr.Button("🧹 Limpiar conversación", variant="secondary")
        with gr.Column(scale=3):
            panel = gr.HTML(panel_inicial())
    gr.HTML(FOOTER)

    send.click(chat_fn, [txt, chatbot], [txt, chatbot, panel])
    txt.submit(chat_fn, [txt, chatbot], [txt, chatbot, panel])
    clear.click(lambda: ("", [], panel_inicial()), None, [txt, chatbot, panel])

demo.launch(share=True, debug=False)

## 14. Ejemplos de preguntas y respuestas

A continuación se muestran intercambios **representativos** del sistema (la salida exacta del
LLM varía en cada ejecución). Tras correr el cuaderno en Colab, **sustituye o complementa**
estos ejemplos con las respuestas reales que obtengas.

---

**👤 Usuaria:** *¿Qué es el ciberacoso?*
**🔎 Detector:** ✅ sin señales claras (prob ≈ 0.10)
**🤖 Seguras:** El ciberacoso es una conducta repetida e intencional que busca humillar,
intimidar o dañar a alguien por medios digitales (redes, mensajería, etc.). Lo que lo
distingue de una discusión puntual es la **repetición** y el **desequilibrio de poder**.
Puede tomar muchas formas: insultos constantes, amenazas, suplantación de identidad o
difusión de información tuya sin permiso. Si algo así te está ocurriendo, no estás sola y
puedo orientarte sobre qué hacer. 💜
*📚 Fuentes: Qué es el ciberacoso · Tipos de ciberacoso · Señales de alerta*

---

**👤 Usuaria:** *Un excompañero me amenaza y dice que va a publicar fotos mías.*
**🔎 Detector:** ⚠️ posible acoso (prob ≈ 0.86)
**🤖 Seguras:** Lamento mucho que estés pasando por esto; **no es tu culpa**. Te sugiero:
1) No respondas a las amenazas. 2) **No borres** los mensajes: guarda capturas con usuario,
fecha y la URL. 3) Bloquéalo y reporta la cuenta en la plataforma. 4) Cuéntale a alguien de
confianza. En México, la **Ley Olimpia** sanciona la difusión de contenido íntimo sin
consentimiento, así que puedes denunciar marcando **079 opción 1** o ante el Ministerio
Público. Si te sientes en riesgo, llama al **911**.
*📚 Fuentes: Marco legal: Ley Olimpia · Qué hacer: resguarda evidencia · Recursos de ayuda en México*

---

**👤 Usuaria:** *Siento que es mi culpa por haber confiado en él...*
**🔎 Detector:** ✅ sin señales claras (prob ≈ 0.22)
**🤖 Seguras:** Quiero que sepas algo importante: **la responsabilidad es de quien acosa, no
tuya.** Confiar en alguien no justifica que te dañen. Lo que sientes es válido y pedir apoyo
es un acto de fuerza, no de debilidad. Considera hablar con una persona de confianza o con
acompañamiento psicológico, y recuerda que la Línea Mujeres **800 108 4053** atiende 24 h. 💜
*📚 Fuentes: Apoyo emocional y autocuidado · Mitos sobre el ciberacoso · Recursos de ayuda en México*


# **Conclusiones**

* **Adecuación de las respuestas.** Al estar **fundamentadas en la base de conocimiento
  (RAG)**, las respuestas se mantienen **pertinentes a la temática** (ciberacoso y violencia
  digital) y consistentes con los recursos reales de México. El sistema deja de dar una
  etiqueta seca («acoso / no acoso») y pasa a **acompañar y orientar**, que es lo que una
  persona en esa situación realmente necesita.

* **Valor del RAG frente a un LLM solo.** El componente de recuperación **reduce las
  alucinaciones**: el LLM ya no improvisa teléfonos o leyes, sino que se apoya en documentos
  curados. Esto es crítico en un dominio sensible donde un dato incorrecto (un número de
  ayuda equivocado) podría tener consecuencias reales.

* **Integración del trabajo previo.** El clasificador de *Seguras1* no se desechó: se
  **reutilizó como herramienta interna** que aporta una señal al chatbot. Así, el proyecto
  evoluciona de un modelo clásico de ML a un sistema conversacional moderno.

* **Cuantización (QLoRA).** Cargar el LLM en **NF4 con doble cuantización y cómputo BF16**
  permitió ejecutar un modelo *instruct* potente en una GPU modesta (T4), conectando
  directamente con la teoría de la sesión.

* **Limitaciones.** (1) El detector se entrenó con tuits, por lo que puede equivocarse con
  textos de otro estilo; por eso se usa solo como *pista*. (2) La base de conocimiento es
  acotada: ampliarla (documentos oficiales, por estado) mejoraría la cobertura. (3) El LLM
  puede seguir cometiendo errores, de ahí los *guardrails* del *prompt* y el aviso de que
  Seguras **no sustituye** atención profesional. (4) La temática exige cuidado **ético**: el
  bot evita juzgar, no promete confidencialidad absoluta y siempre deriva a recursos humanos.

* **Trabajo futuro.** Hacer *fine-tuning* del LLM con **QLoRA** sobre diálogos de orientación,
  ampliar el RAG con fuentes oficiales actualizadas, y agregar detección de riesgo más fina
  para escalar a recursos de emergencia cuando sea necesario.


# **Fin de la actividad chatbot: LLM + RAG** 💜